In [6]:
from viz_utils.tsne import extract_features, get_tsne
from viz_utils.vit_explain import attention_rollout

from transformers import AutoImageProcessor, AutoModel, AutoModelForImageClassification
from datasets import load_dataset
from torch.utils.data import DataLoader
from torchvision import transforms
import torch
from torchvision.models.vision_transformer import vit_b_16, ViT_B_16_Weights

import timm


# Step 1: Load the DINO model and processor
processor = AutoImageProcessor.from_pretrained('facebook/dinov2-large-imagenet1k-1-layer')
# model = AutoModelForImageClassification.from_pretrained('facebook/dinov2-large-imagenet1k-1-layer')
# model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitl14')
# model = vit_b_16(weights=ViT_B_16_Weights.DEFAULT)
model = timm.create_model('vit_base_patch16_224', pretrained=True)

print(processor.image_mean, processor.image_std)

model.eval()

# corruption = "gaussian_noise"
# severity = 1
# Step 2: Load a dataset (e.g., CIFAR-10)
# dataset = load_dataset(f"imagenet-1k", split="validation", streaming=True)
# dataset = load_dataset(f"cansa/Describable-Textures-Dataset-DTD", split="train", streaming=True)
# dataset = load_dataset(f"sxj1215/inaturalist", split="train", streaming=True)
dataset = load_dataset(f"Andron00e/Places365-custom", split="train", streaming=True)

# dataset = load_dataset(f"Tsomaros/ImageNet-C-{corruption}-severity_{severity}", split="validation", streaming=True)

# Step 3: Define preprocessing transformations
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # Resize to the input size expected by DINO
    transforms.ToTensor(),          # Convert PIL Image to PyTorch Tensor
    transforms.Normalize(mean=processor.image_mean, std=processor.image_std)  # Normalize using DINO's mean and std
])


# Step 4: Preprocess dataset and create a DataLoader for batching
def preprocess_image(example):
    # Convert PIL image to tensor and apply transformations
    if "image" in example:
        example["pixel_values"] = transform(example["image"].convert("RGB"))
    else:
        example["pixel_values"] = transform(example["images"][0].convert("RGB"))
    return example


import numpy as np
def collate_fn(examples):
    images = []
    labels = []
    for example in examples:
        if "label" not in list(example.keys()) and "labels" not in list(example.keys()):
            example["label"] = None
        images.append(example["pixel_values"])
        if "label" in example:
            labels.append(example["label"])
        else:
            labels.append(example["labels"])
        
    pixel_values = torch.stack(images)
    labels = torch.full((len(labels),), float('nan'))
    return pixel_values, labels #{"pixel_values": pixel_values, "labels": labels}

# Apply preprocessing on-the-fly using `with_transform`
dataset = dataset.map(preprocess_image)


# Create a DataLoader
dataloader = DataLoader(dataset.with_format("torch"), batch_size=4, collate_fn=collate_fn, shuffle=False)

# Step 5: Perform inference in batches
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

images = []
labels = []
# Inverse normalization for visualization
denormalize = transforms.Normalize(
    mean=[-0.485/0.229, -0.456/0.224, -0.406/0.225],
    std=[1/0.229, 1/0.224, 1/0.225]
)
# import matplotlib.pyplot as plt
# def visualize_batch(real_images, save_path=None):
#     """Visualize a batch of real images and their generated counterparts"""
    
#     # Show a few image pairs
#     # num_to_show = min(4, real_images.size(0))
#     num_to_show = real_images.size(0)
    
#     fig, axes = plt.subplots(2, num_to_show, figsize=(16, 8))
    
#     for i in range(num_to_show):
#         # Convert and denormalize
#         real_img = denormalize(real_images[i].cpu()).permute(1, 2, 0).numpy()

#         print(f"Shape of the images: {real_img.shape}")
#         real_img = np.clip(real_img, 0, 1)
        
#         axes[0, i].imshow(real_img)
#         axes[0, i].set_title(f"Real {i+1}")
#         axes[0, i].axis('off')
            
#     plt.tight_layout()
    
#     if save_path:
#         plt.savefig(save_path)
#         plt.close()
#     else:
#         plt.show()


# batch_num = 1

# desired_class = 0

# for batch in dataloader:
#     saved_flag = False

#     desired_class_indices = (batch['labels'].view(-1,)==desired_class).to(torch.bool)

#     pixel_values = batch["pixel_values"].to(device)
#     labels.extend(list(batch['labels'][desired_class_indices].cpu().numpy().ravel()))

#     images.append(batch['pixel_values'][desired_class_indices])

#     if len(labels)==4:
#         total_images = 0
#         print(len(images))
#         for img in images:
#             total_images += img.size(0)

#         print("Total_images:", total_images)
#         print(f"Length of labels: {len(labels)}")

#         images = torch.cat(images, dim=0)
#         visualize_batch(images, save_path="/home/sazzad/DomainDistillation/src/dino_vis/real_images.png")
#         break
#     else:
#         continue


     

[0.485, 0.456, 0.406] [0.229, 0.224, 0.225]


Resolving data files:   0%|          | 0/322 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/322 [00:00<?, ?it/s]

In [7]:
attention_rollout(model, dataloader, args=None, num_classes=1000, num_image_per_class=1)

/home/sazzad/anaconda3/envs/pytorch-gpu/lib/python3.12/site-packages/datasets/formatting/torch_formatter.py:87: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return torch.tensor(value, **{**default_dtype, **self.torch_tensor_kwargs})


Doing Attention Rollout
Shape of image :  (224, 224, 3)
Shape of heatmap :  (224, 224, 4)
Doing Attention Rollout
Shape of image :  (224, 224, 3)
Shape of heatmap :  (224, 224, 4)
Doing Attention Rollout
Shape of image :  (224, 224, 3)
Shape of heatmap :  (224, 224, 4)
Doing Attention Rollout
Shape of image :  (224, 224, 3)
Shape of heatmap :  (224, 224, 4)
Doing Attention Rollout
Shape of image :  (224, 224, 3)
Shape of heatmap :  (224, 224, 4)
Doing Attention Rollout
Shape of image :  (224, 224, 3)
Shape of heatmap :  (224, 224, 4)
Doing Attention Rollout
Shape of image :  (224, 224, 3)
Shape of heatmap :  (224, 224, 4)
Doing Attention Rollout
Shape of image :  (224, 224, 3)
Shape of heatmap :  (224, 224, 4)
Doing Attention Rollout
Shape of image :  (224, 224, 3)
Shape of heatmap :  (224, 224, 4)
Doing Attention Rollout
Shape of image :  (224, 224, 3)
Shape of heatmap :  (224, 224, 4)


In [8]:
print(processor.image_mean, processor.image_std)

[0.485, 0.456, 0.406] [0.229, 0.224, 0.225]
